# 🚀 Stock Prediction LSTM — Google Colab Trainer

**Setup:** Run cells in order. This notebook will:
1. Mount Google Drive
2. Install dependencies
3. Download project from Drive
4. Train models on GPU
5. Upload results back to Drive

---

## 1️⃣ Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
print('✅ Google Drive mounted')

## 2️⃣ Setup Paths & Download Project

In [ ]:
import os
import subprocess
import zipfile
import shutil

# Create working directory
COLAB_WORK_DIR = '/content/stock-prediction'
GDRIVE_PROJECT_DIR = '/content/gdrive/MyDrive/stock-prediction'
GDRIVE_BACKUPS_DIR = '/content/gdrive/MyDrive/stock-prediction-backups'

os.makedirs(GDRIVE_PROJECT_DIR, exist_ok=True)
os.makedirs(GDRIVE_BACKUPS_DIR, exist_ok=True)

print(f'📁 Working dir: {COLAB_WORK_DIR}')
print(f'📁 Drive dir: {GDRIVE_PROJECT_DIR}')
print(f'📁 Backups dir: {GDRIVE_BACKUPS_DIR}')

## 3️⃣ Install Dependencies

In [ ]:
# Install required packages
!pip install -q yfinance tensorflow scikit-learn pandas matplotlib seaborn joblib -U
print('✅ All dependencies installed')

## 4️⃣ Sync Project From Drive (or first time setup)

In [ ]:
import os
import zipfile
import shutil

# Check if project exists on Drive
zip_path = f'{GDRIVE_PROJECT_DIR}/stock-prediction.zip'

if os.path.exists(zip_path):
    # Extract from Drive
    print(f'📥 Extracting project from Drive...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('/content/')
    print(f'✅ Project extracted to {COLAB_WORK_DIR}')
else:
    print(f'⚠️  No backup zip found. First time setup—will create after training.')

os.chdir(COLAB_WORK_DIR)
print(f'📂 Current dir: {os.getcwd()}')
print(f'📂 Contents: {os.listdir()}')

## 5️⃣ Verify GPU

In [ ]:
import tensorflow as tf

gpu_devices = tf.config.list_physical_devices('GPU')
if gpu_devices:
    print(f'✅ GPU Found: {len(gpu_devices)} device(s)')
    for gpu in gpu_devices:
        print(f'   - {gpu}')
else:
    print('⚠️  No GPU detected. Set Runtime → Change runtime type → GPU')

print(f'\n📊 TensorFlow version: {tf.__version__}')

## 6️⃣ Train Models

In [ ]:
import os
import sys

os.chdir(f'{COLAB_WORK_DIR}/model')
sys.path.insert(0, os.getcwd())

# Train RELIANCE.NS
print('\n' + '='*60)
print('Training RELIANCE.NS...')
print('='*60)

!python train.py \
    --ticker RELIANCE.NS \
    --epochs 200 \
    --window 60 \
    --horizon 1 \
    --batch 32 \
    --lr 0.0003 \
    --dropout 0.15 \
    --output ../backend/models

In [ ]:
# Train TCS.NS
print('\n' + '='*60)
print('Training TCS.NS...')
print('='*60)

!python train.py \
    --ticker TCS.NS \
    --epochs 200 \
    --window 60 \
    --horizon 1 \
    --batch 32 \
    --lr 0.0003 \
    --dropout 0.15 \
    --output ../backend/models

## 7️⃣ Evaluate Models

In [ ]:
import json
from evaluate import full_evaluation

os.chdir(f'{COLAB_WORK_DIR}/model')

results = {}
for ticker in ['RELIANCE.NS', 'TCS.NS']:
    print(f'\n📊 Evaluating {ticker}...')
    report, _, _ = full_evaluation(ticker=ticker, model_dir='../backend/models')
    results[ticker] = report
    print(json.dumps(report, indent=2))

# Save results
eval_file = f'{COLAB_WORK_DIR}/evaluation_results.json'
with open(eval_file, 'w') as f:
    json.dump(results, f, indent=2)
print(f'\n✅ Results saved to {eval_file}')

## 8️⃣ Upload Results to Google Drive

In [ ]:
import os
import shutil
import zipfile
from datetime import datetime

os.chdir(COLAB_WORK_DIR)

# Copy backend/models to Drive
models_src = os.path.join(COLAB_WORK_DIR, 'backend', 'models')
models_dst = os.path.join(GDRIVE_BACKUPS_DIR, f'models_{datetime.now().strftime("%Y%m%d_%H%M%S")}')

if os.path.exists(models_src):
    shutil.copytree(models_src, models_dst, dirs_exist_ok=True)
    print(f'✅ Models backed up to {models_dst}')
else:
    print(f'⚠️  Models dir not found at {models_src}')

# Copy evaluation results
eval_src = os.path.join(COLAB_WORK_DIR, 'evaluation_results.json')
if os.path.exists(eval_src):
    shutil.copy2(eval_src, GDRIVE_BACKUPS_DIR)
    print(f'✅ Evaluation results copied')

# Create project zip for next run
print('📦 Creating project zip for next run...')
zip_path = os.path.join(GDRIVE_PROJECT_DIR, 'stock-prediction.zip')

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, dirs, files in os.walk(COLAB_WORK_DIR):
        # Skip large cache/venv dirs
        dirs[:] = [d for d in dirs if d not in ['.venv', '__pycache__', '.git', 'node_modules']]
        for file in files:
            if not file.endswith(('.pyc', '.pyo', '.pyd')):
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, '/content')
                z.write(file_path, arcname)

print(f'✅ Project zip created: {zip_path}')
print(f'📊 Zip size: {os.path.getsize(zip_path) / 1024 / 1024:.1f} MB')

## ✅ Training Complete!

**What was done:**
- ✅ Models trained with GPU acceleration
- ✅ Results evaluated and saved
- ✅ Models backed up to Google Drive
- ✅ Project snapshot saved for next run

**Next steps on your local machine:**
1. Run the download script (see VS Code integration guide)
2. Models will be synced to `backend/models/`
3. Review results in `evaluation_results.json`